##Step 1: Interpret the output

In [0]:
#Import Pandas library
import pandas as pd

#Load the data into a dataframe
df = spark.read.table("workspace.titantic_machine_learning_assignment.titanic_dataset").toPandas()


# Dataset size: 891 rows — each row represents one passenger, 12 columns — each column represents a variable about the passengers.
print(df.shape)

#Data schema details
df.info()

#This counts the missing values in every column. Age: 177 missing values, Cabin: 687 missing values,Embarked: 2 missing values.
print(df.isnull().sum())

#Calculates the duplicated values in every coloumn, zero found.
print(df.duplicated().sum())

(891, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype:

In [0]:
#Display the first 5 rows of the dataframe
display(df.head(5))

PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.25,null,S
2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Thayer)",female,38.0,1,0,PC 17599,71.2833,C85,C
3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.925,null,S
4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1,C123,S
5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.05,null,S


##Step 2: Check the missing values

In [0]:
#This counts the missing values in every column. Age: 177 missing values, Cabin: 687 missing values,Embarked: 2 missing values.
print(df.isnull().sum())

#Calculating the percentage of the missing values

print((df.isnull().sum() / len(df) * 100).round(2))

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
PassengerId     0.00
Survived        0.00
Pclass          0.00
Name            0.00
Sex             0.00
Age            19.87
SibSp           0.00
Parch           0.00
Ticket          0.00
Fare            0.00
Cabin          77.10
Embarked        0.22
dtype: float64


Let's look at what the percentages mean:

Age	177	19.87%	= Replace with median

Cabin	687	77.10%	=
Exclude from initial model

Embarked 2	0.22%	= Replace with most frequent value

Why these decisions?
- Age: We retain the column because age could help explain passenger survival. The median is less affected by unusually high or low ages than the mean.
- Cabin: Almost 78% of its values are missing. For this beginner exercise, we'll exclude it rather than attempt to reconstruct so much missing information.
- Embarked: Only two values are missing, so replacing them with the most frequent boarding location is reasonable.

##Step 3: Decide how to handle missing values

Can we use passenger information to predict whether a passenger survived the Titanic disaster?
- Target variable (y): Survived
- Features (X): Pclass, Sex, Age, SibSp, Parch, Fare and Embarked.

In [0]:
# Select the columns we want to use
df = df[['Survived', 'Pclass', 'Sex', 'Age',
         'SibSp', 'Parch', 'Fare', 'Embarked']]

# Separate features and target
X = df.drop('Survived', axis=1)
y = df['Survived']

# Check the results
print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (891, 7)
Target shape: (891,)


##Step 4: Split the dataset into training and testing sets

- 80% of our data for training.
- 20% of our data for testing.

In our dataset of 891 passengers, this will give us approximately 712 training records and 179 testing records.

Important: We split before fitting our data-preparation tools. This helps prevent data leakage.

In [0]:
#Divides the dataset into training and testing portions.
from sklearn.model_selection import train_test_split

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #Allocates 20% to testing.
    random_state=42, #Makes the split reproducible.
    stratify=y #Maintains approximately the same survival proportions in both sets
)

# Display the sizes
display("Training features:", X_train.shape) #Training passenger information.
display("Testing features:", X_test.shape) #Testing passenger information.
display("Training target:", y_train.shape) #Correct survival outcomes for training.
display("Testing target:", y_test.shape) #Correct survival outcomes for testing.

'Training features:'

(712, 7)

'Testing features:'

(179, 7)

'Training target:'

(712,)

'Testing target:'

(179,)

##Step 5: Handle missing values

Now we need to deal with the missing values in Age and Embarked.

We'll use a technique called imputation.

Imputation simply means replacing missing values with reasonable substitute values.

We'll use:
- Median for Age.
- Most frequent value (mode) for Embarked.

In [0]:
from sklearn.impute import SimpleImputer

# Create an imputer for numerical columns
num_imputer = SimpleImputer(strategy='median')

# Create an imputer for categorical columns
cat_imputer = SimpleImputer(strategy='most_frequent')

We'll use the median imputer on all five numerical columns and the most-frequent imputer on the two categorical columns. The imputers only change missing values; they leave existing values unchanged.

In [0]:
numeric_columns = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']
categorical_columns = ['Sex', 'Embarked']

# Fit on training data and transform training data
X_train[numeric_columns] = num_imputer.fit_transform(
    X_train[numeric_columns]
)

X_train[categorical_columns] = cat_imputer.fit_transform(
    X_train[categorical_columns]
)

# Transform testing data using the training imputers
X_test[numeric_columns] = num_imputer.transform(
    X_test[numeric_columns]
)

X_test[categorical_columns] = cat_imputer.transform(
    X_test[categorical_columns]
)

What is the difference between fit_transform() and transform()?
- fit_transform() learns the replacement values from the training data and applies them.
- transform() uses those same replacement values on the testing data.
This is important because we don't want information from our test data influencing the training process.

In [0]:

#Check whether missing values remain
print("Training missing values:")
print(X_train.isnull().sum())

print("\nTesting missing values:")
print(X_test.isnull().sum())

print(X_train['Embarked'].value_counts(dropna=False))

print(cat_imputer.statistics_)

Training missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    2
dtype: int64

Testing missing values:
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Embarked
S       516
C       139
Q        55
None      2
Name: count, dtype: int64
['male' 'S']


In [0]:
# Replace None with NaN in Embarked
X_train['Embarked'] = X_train['Embarked'].replace({None: float('nan')})

# Apply the categorical imputer again
X_train[categorical_columns] = cat_imputer.fit_transform(
    X_train[categorical_columns]
)

X_test[categorical_columns] = cat_imputer.transform(
    X_test[categorical_columns]
)

print(X_train.isnull().sum())
print(X_test.isnull().sum())

Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


Both your training and testing datasets now have zero missing values.

This means we've successfully handled the missing data using:
- Median imputation for numerical columns.
- Most-frequent-value imputation for categorical columns.

##Step 6: Encode categorical variables

We need to convert these categories into numbers.

We'll use a technique called One-Hot Encoding

- fit_transform() learns the categories from the training data and converts them into numerical columns.
- transform() applies the same encoding to the testing data.
- get_feature_names_out() shows us the names of the new columns.

In [0]:

#import the encoder
from sklearn.preprocessing import OneHotEncoder
#create the encoder
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

#Encode Sex and Embarked

# Fit the encoder on training data
X_train_encoded = encoder.fit_transform(
    X_train[categorical_columns]
)

# Transform testing data
X_test_encoded = encoder.transform(
    X_test[categorical_columns]
)

print("Encoded training data:")
print(X_train_encoded[:5])

print("\nEncoded column names:")
print(encoder.get_feature_names_out(categorical_columns))

Encoded training data:
[[0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [0. 1. 0. 0. 1.]
 [1. 0. 0. 0. 1.]
 [1. 0. 0. 0. 1.]]

Encoded column names:
['Sex_female' 'Sex_male' 'Embarked_C' 'Embarked_Q' 'Embarked_S']


##Step 7: Scale numerical variables

- fit_transform() calculates the scaling information using training data and scales it.
- transform() applies the same scaling information to testing data.
- We never fit the scaler separately on the test data.
This prevents data leakage.

In [0]:
# Import standard scaler

from sklearn.preprocessing import StandardScaler

#Create scaler
scaler = StandardScaler()

#Scale 5 numerical columns

numeric_columns = ['Age', 'Fare', 'Pclass', 'SibSp', 'Parch']

# Fit on training data and transform
X_train_scaled = scaler.fit_transform(
    X_train[numeric_columns]
)

# Transform testing data
X_test_scaled = scaler.transform(
    X_test[numeric_columns]
)

#Inspecct the scaled results

print("Scaled training data:")
print(X_train_scaled[:5])

Scaled training data:
[[-0.08113533  0.5138115   0.82956755 -0.46508428 -0.46618317]
 [-0.08113533 -0.66256323 -0.37094484 -0.46508428 -0.46618317]
 [-0.08113533  3.95539858 -1.57145722 -0.46508428 -0.46618317]
 [-0.88782719 -0.46787435  0.82956755 -0.46508428  0.72778236]
 [ 0.11093416 -0.11597681 -0.37094484  0.47833454  0.72778236]]


##Step 8: Combine the prepared data

In [0]:
import numpy as np

# Combine scaled numerical and encoded categorical features
X_train_final = np.concatenate(
    [X_train_scaled, X_train_encoded], axis=1
)

X_test_final = np.concatenate(
    [X_test_scaled, X_test_encoded], axis=1
)

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (712, 10)
Final testing shape: (179, 10)


##Step 9: Identify outliers

What is an outlier?
An outlier is a data value that is unusually high or unusually low compared with most other values.Use the IQR method
We'll use the Interquartile Range (IQR) method to identify possible outliers. 
Apply the method to Age and Fare.

Let's use the original numerical values in X_train, before scaling.

In [0]:
# Select columns to investigate
columns_to_check = ['Age', 'Fare']

for column in columns_to_check:

    Q1 = X_train[column].quantile(0.25)
    Q3 = X_train[column].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = X_train[
        (X_train[column] < lower_bound) |
        (X_train[column] > upper_bound)
    ]

    print(f"\n{column}")
    print("Lower boundary:", lower_bound)
    print("Upper boundary:", upper_bound)
    print("Number of potential outliers:", len(outliers))


Age
Lower boundary: 1.0
Upper boundary: 57.0
Number of potential outliers: 29

Fare
Lower boundary: -26.7605
Upper boundary: 65.6563
Number of potential outliers: 91


##Step 10: Feature engineering

We're combining two existing features:
- SibSp: Number of siblings and spouses aboard.
- Parch: Number of parents and children aboard.

In [0]:
# Create a new feature
X_train['FamilySize'] = (
    X_train['SibSp'] + X_train['Parch'] + 1
)

X_test['FamilySize'] = (
    X_test['SibSp'] + X_test['Parch'] + 1
)

# Display the results
print(X_train[['SibSp', 'Parch', 'FamilySize']].head())

     SibSp  Parch  FamilySize
692    0.0    0.0         1.0
481    0.0    0.0         1.0
527    0.0    0.0         1.0
855    0.0    1.0         2.0
801    1.0    1.0         3.0


## Step 11: Update your final prepared dataset

In [0]:
# Include FamilySize in the numerical features
numeric_columns = [
    'Age', 'Fare', 'Pclass', 'SibSp', 'Parch', 'FamilySize'
]

# Create a new scaler
scaler_final = StandardScaler()

# Scale training data
X_train_scaled_final = scaler_final.fit_transform(
    X_train[numeric_columns]
)

# Scale testing data using training statistics
X_test_scaled_final = scaler_final.transform(
    X_test[numeric_columns]
)

# Combine numerical and encoded categorical features
X_train_final = np.concatenate(
    [X_train_scaled_final, X_train_encoded], axis=1
)

X_test_final = np.concatenate(
    [X_test_scaled_final, X_test_encoded], axis=1
)

print("Final training shape:", X_train_final.shape)
print("Final testing shape:", X_test_final.shape)

Final training shape: (712, 11)
Final testing shape: (179, 11)
